# Snow rate exploration — liquid input vs. raw precipitation

**Purpose:** visualize how *liquid input* (rain + degree-day melt) diverges in
timing from raw precipitation at real HiRO-ACE grid points — winter
accumulation, spring release — to motivate
[`exploration/SWE/docs/snow_rate.md`](../docs/snow_rate.md)'s **Method A2**
(degree-day snow preprocessor) before touching any model code. Section 6.2 of
that doc's source paper identifies mistimed spring snowmelt as the main
remaining natural-forcing error in snow-dominated basins — this notebook
makes that mistiming visible at a handful of real grid points.

**Everything runs in this one notebook** — DEM-based grid-point selection,
lapse-rate temperature downscaling, the degree-day recursion, and all plots —
following this repo's existing convention for exploration notebooks (see
`processing/temp_downscaling/scripts/lapse_rate.ipynb`, which does the same
kind of interactive walkthrough for the downscaling step alone). Downscaling
here is done **in-memory** against a handful of point columns for one 8-month
window — not the full 10-year, full-Japan-grid batch job — so it should be
light enough to run directly, without a separate sbatch submission. If it
still trips the login node's thread limit (hit once before, on the much
heavier full-grid job), fall back to running just the downscaling cells via
`processing/temp_downscaling/scripts/run_downscaling.py` as a batch job and
re-loading the (small) result into this notebook.

**Method A2 recap** (fixed, illustrative parameters — not tuned):

```
if T <= T_thresh:  snowfall = P;  rain = 0
else:              snowfall = 0;  rain = P

melt = min(SWE, DDF * max(0, T - T_melt))
SWE  = SWE + snowfall - melt
liquid_input = rain + melt
```

`T_thresh = T_melt = 0 °C`, `DDF ≈ 3 mm/°C/day` (Hock 2003's typical
temperature-index range is ~2–5 mm/°C/day). This notebook uses the **daily**
recursion the doc specifies (§4) — 6-hourly forcing is aggregated to daily
(temp: mean, precip: sum) before the loop.

**Data:** `processing/temp_downscaling` only (temporal_binning and
catchment_weighting are skipped — this stays at the grid level, not
catchment means), initial condition `ic0000`, window **Nov 2014 – Jun 2015**,
native 6-hourly cadence.

> **Written offline, unverified against real data.** Isambard wasn't
> reachable while this notebook was authored, so the config constants below
> (paths, variable names, dimension names) are inferred from other scripts/
> docs in this repo (`processing/scripts/isambard/run_smoke_test.sh`,
> `processing/catchment_weighting/docs/catchment_weighting.md`,
> `processing/temp_downscaling/scripts/lapse_rate_lib.py`) but **not yet run
> end-to-end**. The first code cell after the config block does a quick
> `ds.dims` / `ds.data_vars` sanity check — fix the config there before
> trusting anything downstream.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import sys
sys.path.insert(0, str(Path("../../../processing/temp_downscaling/scripts").resolve()))
from lapse_rate_lib import (
    DEFAULT_LAPSE_RATE, bbox_from_grid, build_high_res_dem,
    load_low_res_dem, load_target_grid, lapse_rate_correct, to_celsius,
)

# --- Isambard paths (mirrors processing/scripts/isambard/run_smoke_test.sh) --
BASE = Path("/projects/u6t/vbrekke/climate-hydro-pipeline")
IC = "ic0000"
ACE2S_ZARR = BASE / f"hiroace/outputs/with_temp/ace2s/output_6hourly_ace2s_{IC}.zarr"
HIRO_ZARR = BASE / f"hiroace/outputs/with_temp/hiro/Japan_10yr_20140101_20231231_{IC}.zarr"
FORCING_NC = BASE / "hiroace/data/forcing_data/forcing_2023.nc"
DEM_CACHE = BASE / "processing/temp_downscaling/dem_cache/etopo2022_15s_japan.nc"

# --- window (native 6-hourly, no rebinning) ----------------------------------
TIME_START, TIME_END = "2014-11-01", "2015-06-30"

# --- variable / dimension names -- VERIFY against the real stores -----------
# ace2s (low-res temperature): dims (time, lat, lon) per lapse_rate_lib's own
# defaults (lat_name="lat"/lon_name="lon" for the *_zarr helpers) -- unlike
# forcing_nc's HGTsfc, which load_low_res_dem renames from latitude/longitude.
TEMP_VAR = "TMP2m"
ACE2S_LAT, ACE2S_LON = "lat", "lon"
# HiRO (precip + the target high-res grid): dims (time, [ensemble,] latitude,
# longitude) per processing/catchment_weighting/docs/catchment_weighting.md.
PRECIP_VAR = "PRATEsfc"
HIRO_LAT, HIRO_LON = "latitude", "longitude"

# --- degree-day parameters (Method A2, fixed & illustrative -- see doc) -----
T_THRESH = 0.0   # deg C, rain/snow partition threshold
T_MELT = 0.0     # deg C, melt threshold
DDF = 3.0        # mm / degC / day, degree-day factor (Hock 2003: ~2-5)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                      "axes.spines.top": False, "axes.spines.right": False})

# Fixed categorical palette, reused consistently across every plot below.
COLOR_PRECIP = "#0072B2"        # blue
COLOR_LIQUID_INPUT = "#D55E00"  # vermillion
COLOR_SWE = "#7570B3"           # purple
COLOR_ZERO_LINE = "#888888"     # neutral gray, reference not data
COLOR_TEMP = "#333333"          # near-black, single series

### Sanity check — confirm the config above against the real stores

Run this before anything else once Isambard is reachable. It should print
each store's dims and data vars; if `TEMP_VAR`/`PRECIP_VAR`/dim names above
don't match, fix the config cell and re-run from the top.

In [ ]:
for name, path in [("ACE2S", ACE2S_ZARR), ("HIRO", HIRO_ZARR)]:
    ds = xr.open_zarr(path)
    print(f"--- {name}: {path} ---")
    print("dims:", dict(ds.dims))
    print("data_vars:", list(ds.data_vars))
    print()

assert DEM_CACHE.exists(), f"DEM cache missing: {DEM_CACHE} (see processing/temp_downscaling/docs)"
assert FORCING_NC.exists(), f"Forcing NetCDF missing: {FORCING_NC}"
print("DEM cache and forcing NetCDF found.")

## 1. Grid point selection — derived from the DEM, not eyeballed

Use the already-cached ETOPO 2022 DEM, block-averaged onto HiRO's own
high-res grid (`build_high_res_dem` — this reuses the cache, so it's cheap;
no re-download), to find the highest-elevation cell inside each of three
snowy regions (Sea-of-Japan-side Tohoku, Hokkaido, the Chubu/Japan Alps),
plus one non-snowy lowland point near Nagoya as a contrast baseline.

Region boxes are approximate (drawn around known heavy-snow / high-alpine
areas) — the *point* itself is whichever real grid cell has max elevation
inside the box, not a hand-picked coordinate. Section 2 below confirms each
snowy candidate's downscaled winter temperature actually crosses 0 °C; if one
doesn't, widen/move its box and re-run rather than accepting it as-is.

In [ ]:
target_lat, target_lon = load_target_grid(HIRO_ZARR, HIRO_LAT, HIRO_LON)
full_bbox = bbox_from_grid(target_lat, target_lon, pad=1.0)

# Cheap: DEM_CACHE already holds the fetched ETOPO subset, this is just a
# block-mean regrid (numpy binning) over the full Japan grid, not a download.
elevation, _elev_std = build_high_res_dem(target_lat, target_lon, cache_path=DEM_CACHE)
elevation = elevation.rename({"lat": "latitude", "lon": "longitude"})

REGIONS = {
    # name -> (lat_min, lat_max, lon_min, lon_max)
    "tohoku_seaofjapan": (38.0, 40.3, 139.6, 140.9),   # Ou mountains (Zao/Chokai/Gassan)
    "hokkaido":          (43.2, 43.8, 142.7, 143.2),   # Daisetsuzan (Mt. Asahidake ~2291m)
    "chubu_japan_alps":  (35.9, 36.6, 137.4, 137.9),   # Hida mountains (Yari/Hotaka)
}
LOWLAND_REF = ("nagoya_lowland", 35.18, 136.91)  # non-snowy contrast baseline

def argmax_point_in_bbox(elev, lat_min, lat_max, lon_min, lon_max):
    sub = elev.sel(latitude=slice(lat_min, lat_max), longitude=slice(lon_min, lon_max))
    flat_idx = int(sub.values.argmax())
    iy, ix = np.unravel_index(flat_idx, sub.shape)
    return float(sub["latitude"].values[iy]), float(sub["longitude"].values[ix]), float(sub.values[iy, ix])

points = {}
for name, (lat_min, lat_max, lon_min, lon_max) in REGIONS.items():
    lat, lon, elev = argmax_point_in_bbox(elevation, lat_min, lat_max, lon_min, lon_max)
    points[name] = dict(lat=lat, lon=lon, elev_m=elev, kind="snowy_candidate")

name, ref_lat, ref_lon = LOWLAND_REF
pt = elevation.sel(latitude=ref_lat, longitude=ref_lon, method="nearest")
points[name] = dict(lat=float(pt["latitude"]), lon=float(pt["longitude"]),
                     elev_m=float(pt.values), kind="lowland_baseline")

points_df = pd.DataFrame(points).T
points_df[["lat", "lon", "elev_m"]] = points_df[["lat", "lon", "elev_m"]].astype(float)
points_df

### Locator map (sanity check on the picks)

Elevation as a single sequential ramp (magnitude, not identity — no rainbow
terrain colormap), candidate points overlaid.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 7))
mesh = ax.pcolormesh(elevation["longitude"], elevation["latitude"], elevation.values,
                      cmap="Blues", shading="auto", vmin=0)
fig.colorbar(mesh, ax=ax, label="Elevation (m)", shrink=0.8)

for name, row in points_df.iterrows():
    marker = "^" if row["kind"] == "snowy_candidate" else "o"
    ax.scatter(row["lon"], row["lat"], s=90, marker=marker,
               facecolor=COLOR_LIQUID_INPUT, edgecolor="white", linewidth=1.2, zorder=5)
    ax.annotate(name, (row["lon"], row["lat"]), textcoords="offset points",
                xytext=(6, 6), fontsize=9, color="#222")

ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Candidate grid points (▲ snowy candidate, ● lowland baseline)")
ax.set_aspect("equal")
fig.tight_layout()

## 2. Downscale temperature at each point (lapse-rate correction, in-memory)

Only ACE2S's coarse `TMP2m`, loaded for a bounding box wide enough to give
`RegularGridInterpolator` real neighbouring low-res cells around every point,
and for the Nov 2014 – Jun 2015 window only. That's a few hundred low-res
cells × ~960 timesteps — small enough to `.load()` outright, unlike the
full-Japan / full-10-year job that hit the login node's thread limit earlier.

Precipitation is **not** downscaled here — it's read directly off HiRO's own
already-high-res grid in §4 (lapse-rate correction is temperature-only; see
`processing/temp_downscaling/docs/lapse_rate_downscaling.md`).

In [ ]:
# Bbox covering every candidate point with generous padding for the low-res
# (~1 deg) ACE2S interpolation neighbourhood.
pts_lat_pad = (points_df["lat"].min() - 3.0, points_df["lat"].max() + 3.0)
pts_lon_pad = (points_df["lon"].min() - 3.0, points_df["lon"].max() + 3.0)
points_bbox = dict(lat_min=pts_lat_pad[0], lat_max=pts_lat_pad[1],
                    lon_min=pts_lon_pad[0], lon_max=pts_lon_pad[1])

z_low = load_low_res_dem(FORCING_NC, bbox=points_bbox)  # HGTsfc, renamed to lat/lon

t_low_raw = (
    xr.open_zarr(ACE2S_ZARR)[TEMP_VAR]
    .rename({ACE2S_LAT: "lat", ACE2S_LON: "lon"})
    .sel(lat=slice(points_bbox["lat_min"], points_bbox["lat_max"]),
         lon=slice(points_bbox["lon_min"], points_bbox["lon_max"]))
    .sel(time=slice(TIME_START, TIME_END))
    .load()
)
t_low = to_celsius(t_low_raw)
print(f"t_low: {dict(t_low.sizes)}, {t_low['time'].min().values} .. {t_low['time'].max().values}")

In [ ]:
temp_series = {}  # point name -> pd.Series (6-hourly, deg C), indexed by time

for name, row in points_df.iterrows():
    z_high_point = xr.DataArray(
        [[row["elev_m"]]], dims=("lat", "lon"),
        coords={"lat": [row["lat"]], "lon": [row["lon"]]},
    )
    corrected = lapse_rate_correct(t_low, z_low, z_high_point, lapse_rate=DEFAULT_LAPSE_RATE)
    temp_series[name] = corrected.squeeze(("lat", "lon"), drop=True).to_series()

temp_df_6h = pd.DataFrame(temp_series)
temp_df_6h.head()

## 3. Winter validation — do the snowy candidates actually freeze?

Aggregate to daily (mean temp) and check each `snowy_candidate` point's
minimum daily-mean winter temperature is comfortably below 0 °C. The day
boundary follows this pipeline's own end-labeled 6-hourly convention (a
day's samples are its 06:00, 12:00, 18:00, and following 00:00 — see
`processing/temporal_binning`'s day-binning notes) — i.e. shift back 6h
before flooring to a calendar date, so the 00:00 sample lands on the day it
closes rather than the day it opens.

In [ ]:
def to_daily_mean(series_6h):
    day = (series_6h.index - pd.Timedelta(hours=6)).floor("D")
    return series_6h.groupby(day).mean()

temp_df_daily = temp_df_6h.apply(to_daily_mean)

winter = temp_df_daily.loc["2014-12-01":"2015-02-28"]
print("Winter (DJF) daily-mean temperature, deg C:")
print(winter.describe().loc[["min", "mean", "max"]].T)

for name, row in points_df.iterrows():
    if row["kind"] != "snowy_candidate":
        continue
    tmin = winter[name].min()
    status = "OK (freezes)" if tmin < -2 else "CHECK -- barely/doesn't freeze, widen its region bbox"
    print(f"  {name}: winter min = {tmin:.1f} degC -> {status}")

## 4. Load precipitation at the same points (native HiRO grid, no downscaling)

`PRATEsfc` is a 6-hour window-mean **rate**, end-labeled (matches
`processing/temporal_binning`'s convention). Depth over a window = rate ×
window length in hours — the same construction
`processing/scripts/isambard/check_smoke_test.py` uses for its precip
mass-conservation check. Daily depth (mm) = sum of the day's four 6-hourly
depths.

In [ ]:
precip_raw = xr.open_zarr(HIRO_ZARR)[PRECIP_VAR]
if "ensemble" in precip_raw.dims:
    # This IC-specific 10yr store may or may not carry the 4-member ensemble
    # dim seen in the 2-step demo file -- if present, first member only.
    precip_raw = precip_raw.isel(ensemble=0)
    print("NOTE: PRATEsfc had an 'ensemble' dim -- using member 0.")

precip_6h_rate = {}
for name, row in points_df.iterrows():
    da = precip_raw.sel({HIRO_LAT: row["lat"], HIRO_LON: row["lon"]}, method="nearest")
    precip_6h_rate[name] = da.sel(time=slice(TIME_START, TIME_END)).load().to_series()

precip_df_6h_rate = pd.DataFrame(precip_6h_rate)
precip_df_6h_depth = precip_df_6h_rate * 6.0  # rate * 6h window -> mm over that window

def to_daily_sum(series_6h_depth):
    day = (series_6h_depth.index - pd.Timedelta(hours=6)).floor("D")
    return series_6h_depth.groupby(day).sum()

precip_df_daily = precip_df_6h_depth.apply(to_daily_sum)
precip_df_daily.head()

## 5. Degree-day recursion (Method A2, fixed parameters)

Plain sequential Python loop — `SWE_t` depends on `SWE_{t-1}`, and this is
~240 days × a handful of points, trivial cost. `T_thresh = T_melt = 0 °C`,
`DDF = 3 mm/°C/day`, both fixed and illustrative (see notebook intro).

In [ ]:
def degree_day_recursion(temp_c, precip_mm, t_thresh=T_THRESH, t_melt=T_MELT, ddf=DDF):
    '''temp_c, precip_mm: aligned daily pd.Series (same DatetimeIndex).
    Returns a DataFrame with columns temp, precip, snowfall, rain, melt, SWE,
    liquid_input -- one row per day, SWE starting at 0.
    '''
    idx = temp_c.index
    snowfall = np.zeros(len(idx)); rain = np.zeros(len(idx))
    melt = np.zeros(len(idx)); swe = np.zeros(len(idx))

    swe_prev = 0.0
    for i, (t, p) in enumerate(zip(temp_c.values, precip_mm.values)):
        if t <= t_thresh:
            snowfall[i], rain[i] = p, 0.0
        else:
            snowfall[i], rain[i] = 0.0, p
        melt[i] = min(swe_prev, ddf * max(0.0, t - t_melt))
        swe_prev = swe_prev + snowfall[i] - melt[i]
        swe[i] = swe_prev

    out = pd.DataFrame({"temp": temp_c.values, "precip": precip_mm.values,
                         "snowfall": snowfall, "rain": rain, "melt": melt, "SWE": swe},
                        index=idx)
    out["liquid_input"] = out["rain"] + out["melt"]
    return out

results = {}
for name in points_df.index:
    t = temp_df_daily[name].dropna()
    p = precip_df_daily[name].reindex(t.index).fillna(0.0)  # align, defensive
    results[name] = degree_day_recursion(t, p)

results[points_df.index[0]].head()

## 6. Plots — per point

(a) daily temperature with the 0 °C threshold, (b) daily precipitation vs.
liquid input overlaid, (c) SWE storage over time, (d) cumulative precip vs.
cumulative liquid input. One fixed color per series across every point's
figure (blue = precip, vermillion = liquid input, purple = SWE) so points are
visually comparable.

In [ ]:
def plot_point(name, df):
    fig, axes = plt.subplots(4, 1, figsize=(9, 11), sharex=True)
    fig.suptitle(f"{name}  (elev {points_df.loc[name, 'elev_m']:.0f} m, "
                 f"{points_df.loc[name, 'kind']})", fontsize=12)

    # (a) temperature
    ax = axes[0]
    ax.plot(df.index, df["temp"], color=COLOR_TEMP, linewidth=1.4, label="Daily mean temp")
    ax.axhline(0, color=COLOR_ZERO_LINE, linewidth=1.2, linestyle="--", label="0 °C")
    ax.fill_between(df.index, df["temp"], 0, where=(df["temp"] < 0),
                     color=COLOR_ZERO_LINE, alpha=0.12, zorder=0)
    ax.set_ylabel("Temp (°C)")
    ax.legend(loc="upper left", frameon=False, fontsize=8)

    # (b) precip vs liquid input, overlaid, one y-axis (both are mm/day depths)
    ax = axes[1]
    ax.plot(df.index, df["precip"], color=COLOR_PRECIP, linewidth=1.1, alpha=0.85, label="Precipitation")
    ax.plot(df.index, df["liquid_input"], color=COLOR_LIQUID_INPUT, linewidth=1.1, alpha=0.9, label="Liquid input")
    ax.set_ylabel("mm/day")
    ax.legend(loc="upper left", frameon=False, fontsize=8)

    # (c) SWE storage
    ax = axes[2]
    ax.fill_between(df.index, df["SWE"], 0, color=COLOR_SWE, alpha=0.25)
    ax.plot(df.index, df["SWE"], color=COLOR_SWE, linewidth=1.4, label="SWE")
    ax.set_ylabel("SWE (mm)")
    ax.legend(loc="upper left", frameon=False, fontsize=8)

    # (d) cumulative precip vs cumulative liquid input
    ax = axes[3]
    ax.plot(df.index, df["precip"].cumsum(), color=COLOR_PRECIP, linewidth=1.6, label="Cumulative precipitation")
    ax.plot(df.index, df["liquid_input"].cumsum(), color=COLOR_LIQUID_INPUT, linewidth=1.6, label="Cumulative liquid input")
    ax.set_ylabel("mm"); ax.set_xlabel("Date")
    ax.legend(loc="upper left", frameon=False, fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

    for a in axes:
        a.grid(True, alpha=0.25)
    fig.tight_layout()
    return fig

for name, df in results.items():
    plot_point(name, df)

## 7. Summary metrics — how much does the liquid-input reshaping matter?

- **Peak SWE** and the date it occurs — how much winter precip gets locked up.
- **% of winter precip delayed** into the snowpack (peak SWE ÷ total precip).
- **Day of 50% cumulative** precip vs. liquid input, and the gap between them
  in days — the actual timing shift Method A2 is meant to fix.
- **Max daily melt rate** (mm/day) — sanity check against `DDF` (should never
  exceed `DDF * max daily positive-degree-day`, i.e. bounded by physics, not
  by the SWE cap, except late in melt-out).

In [ ]:
def summarize(name, df):
    total_precip = df["precip"].sum()
    total_liquid = df["liquid_input"].sum()
    peak_swe = df["SWE"].max()
    peak_swe_date = df["SWE"].idxmax()

    def day_of_pct(series, pct):
        cum = series.cumsum()
        total = cum.iloc[-1]
        if total <= 0:
            return pd.NaT
        return cum[cum >= pct * total].index[0]

    d50_precip = day_of_pct(df["precip"], 0.5)
    d50_liquid = day_of_pct(df["liquid_input"], 0.5)
    lag_days = (d50_liquid - d50_precip).days if pd.notna(d50_liquid) and pd.notna(d50_precip) else np.nan

    return dict(
        total_precip_mm=total_precip, total_liquid_input_mm=total_liquid,
        peak_SWE_mm=peak_swe, peak_SWE_date=peak_swe_date,
        pct_precip_delayed=100 * peak_swe / total_precip if total_precip > 0 else np.nan,
        day50_precip=d50_precip, day50_liquid_input=d50_liquid, lag_days=lag_days,
        max_daily_melt_mm=df["melt"].max(),
    )

summary_df = pd.DataFrame({name: summarize(name, df) for name, df in results.items()}).T
summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
order = summary_df["lag_days"].astype(float).sort_values().index
ax.bar(order, summary_df.loc[order, "lag_days"].astype(float), color=COLOR_LIQUID_INPUT, width=0.5)
ax.set_ylabel("Lag: day-of-50%-liquid-input − day-of-50%-precip (days)")
ax.set_title("Spring release lag introduced by the snow store, per point")
ax.axhline(0, color=COLOR_ZERO_LINE, linewidth=1)
for label in ax.get_xticklabels():
    label.set_rotation(20); label.set_ha("right")
fig.tight_layout()

## Notes / caveats

- `T_thresh`, `T_melt`, `DDF` are fixed, illustrative values, not fit to any
  observation — see `exploration/SWE/docs/snow_rate.md` §4–5. The point here
  is to show the *shape* of the timing shift, not to validate melt rates.
- The lowland baseline point should show `lag_days ≈ 0` and negligible
  peak SWE — if it doesn't, its region pick landed somewhere snowier than
  intended and should be moved.
- Everything above uses one ensemble/IC member (`ic0000`) and one winter —
  not a claim about the general magnitude of the effect, just a
  proof-of-timing-shift on real data.